<a href="https://colab.research.google.com/github/Aayushi1402/NATURAL-LANGUAGE-PROCESSING/blob/main/Hybrid_Probabilistic_%2B_FSM_Based_Sequence_Modeling_for_document_boundary_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Hybrid Probabilistic + FSM-Based Sequence Modeling
# for Document Boundary Detection

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Sample pages with document boundary labels
pages = [
    "INVOICE No 101 Total Amount 5000",
    "Payment due date 20 December",
    "REPORT Annual Sales Performance",
    "Revenue increased by 20 percent",
    "INVOICE No 102 Total Amount 3000",
    "Payment received successfully"
]

# Labels: 1 = New document, 0 = Continuation
labels = [1, 0, 1, 0, 1, 0]

# Train probabilistic classifier
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(pages)

classifier = MultinomialNB()
classifier.fit(X, labels)

# FSM states
START = "START"
DOCUMENT = "DOCUMENT"

state = START
documents = []
current_document = []

# Process pages sequentially
for page in pages:
    features = vectorizer.transform([page])
    probability = classifier.predict_proba(features)[0]
    new_doc_probability = probability[list(classifier.classes_).index(1)]

    # Boundary detection using probability and FSM rules
    has_heading = bool(
        re.search(r'\b(INVOICE|REPORT|LETTER|CONTRACT)\b',
                  page, re.IGNORECASE)
    )

    if current_document and (
        new_doc_probability >= 0.5 or has_heading
    ):
        documents.append(current_document)
        current_document = []
        state = START

    current_document.append(page)
    state = DOCUMENT

if current_document:
    documents.append(current_document)

# Display detected documents
print("Detected Documents:\n")

for i, document in enumerate(documents, start=1):
    print(f"Document {i}:")
    for page in document:
        print(" -", page)
    print()

Detected Documents:

Document 1:
 - INVOICE No 101 Total Amount 5000
 - Payment due date 20 December

Document 2:
 - REPORT Annual Sales Performance
 - Revenue increased by 20 percent

Document 3:
 - INVOICE No 102 Total Amount 3000
 - Payment received successfully

